In [3]:
from pathlib import Path
from zipfile import ZipFile
from io import TextIOWrapper
from gensim import corpora
from gensim.models import LdaModel

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

import spacy
import re
import random
import pyLDAvis
import pyLDAvis.gensim_models

ModuleNotFoundError: No module named 'gensim'

In [4]:
import sys
print(sys.executable)

/opt/python/bin/python


# Chargement des données

Nous ne travaillons que sur les données de 1993.

In [4]:
zip_path = Path("../data/legislatives_1993.zip")
print(zip_path.exists())

True


In [5]:

texts = []

with ZipFile(zip_path, 'r') as zf:
    for name in zf.namelist():

        # On ne garde que les vrais fichiers .txt (évite les dossiers)
        if name.lower().endswith(".txt"):
            try:
                with zf.open(name) as f:
                    try:
                        txt = TextIOWrapper(f, encoding="utf-8", errors="replace").read()
                    except UnicodeDecodeError:
                        # fallback latin-1
                        f.seek(0)
                        txt = TextIOWrapper(f, encoding="latin-1", errors="replace").read()

                texts.append(txt)

            except Exception as e:
                print(f" Erreur sur {name} : {e}")

print(f"{len(texts)} articles chargés")

5936 articles chargés


In [6]:
# Premier article
print(len(texts[0]))
print(texts[0]) 
print("********************************************")


5527
Département de Seine-Maritime - 12ème Circonscription - Scrutin du 21 Mars 1993
Alain LE VERN Député Maire de Saint-Saëns - 44 ans Suppléant : Docteur Christian PLAILLY Maire de GAILLEFONTAINE - 44 ans
Chère Madame, Chère Mademoiselle, Cher Monsieur,
Voici 5 ans vous m'avez élu Député. J'ai depuis consacré tout mon temps et toute mon énergie pour être digne de votre confiance. La période électorale est trop souvent celle des divisions, des belles promesses ... Jugez les actes, les faits ! Nous n'avons pas tout réussi, bien sûr, mais j'ai travaillé pour rassembler et unir dans l'intérêt de notre circonscription, de ses habitants, de notre Pays.
DANS NOTRE CIRCONSCRIPTION, J'AI AGI POUR: ☒ La construction de logements (Gournay, La Feuillie, Buchy, Saint-Saëns, Neuville Ferrières, Neufchâtel, Londinières, Blangy, Torcy le Grand, Bellencombre ... une centaine par an au lieu d'une dizaine par an avant !). ☒ Le désenclavement routier : RN 27 Rouen Dieppe, Autoroute A 28 ouverte en décem

In [7]:
# Deuxième article
print(len(texts[1]))
print(texts[1])
print("********************************************")

4293
ELECTIONS LEGISLATIVES DU 21 MARS 1993
REPUBLIQUE FRANÇAISE - 2™e CIRCONSCRIPTION DE LA DORDOGNE
Michel SUCHOD Député du Bergeracois
Suppléant
François LASTERNAS Conseiller général de La Force Maire de Prigonrieux
DEUX SOCIALISTES POUR LA RELEVE DE LA GAUCHE DEUX HOMMES EFFICACES POUR LE BERGERACOIS
Madame, Mademoiselle, Monsieur,
Comme vous je suis citoyen.
Et si comme vous, je pense que le gouvernement sortant ne rend pas copie blanche, je pense comme vous qu'il n'a pas su régler le dramatique problème de l'emploi, et qu'il a laissé se développer des « affaires » trop nombreuses et particulièrement nauséabondes.
Il faut donc changer la politique, pour recréer l'emploi, raffermir la démocratie, sauver la paix.
Cela implique-t-il de liquider le député sortant du Bergeracois pour le remplacer par la représen- tante de la droite conservatrice comme certains le souhaitent ? Poser la question, c'est y répondre par la négative.
Bien mieux, il faut garder Michel SUCHOD, le député qui :


# Nettoyage

In [8]:

def clean_text(text):
    
    text = text.lower()
    
    # supprimer mentions archives
    text = re.sub(r"sciences po / fonds cevipof", "", text)
    
    # supprimer caractères spéciaux
    text = re.sub(r"[☒•«»]", " ", text)
    
    # supprimer chiffres
    text = re.sub(r"\d+", " ", text)
    
    # supprimer ponctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # supprimer espaces multiples
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

clean_texts = [clean_text(t) for t in texts]
print(clean_texts[0])

département de seine maritime ème circonscription scrutin du mars alain le vern député maire de saint saëns ans suppléant docteur christian plailly maire de gaillefontaine ans chère madame chère mademoiselle cher monsieur voici ans vous m avez élu député j ai depuis consacré tout mon temps et toute mon énergie pour être digne de votre confiance la période électorale est trop souvent celle des divisions des belles promesses jugez les actes les faits nous n avons pas tout réussi bien sûr mais j ai travaillé pour rassembler et unir dans l intérêt de notre circonscription de ses habitants de notre pays dans notre circonscription j ai agi pour la construction de logements gournay la feuillie buchy saint saëns neuville ferrières neufchâtel londinières blangy torcy le grand bellencombre une centaine par an au lieu d une dizaine par an avant le désenclavement routier rn rouen dieppe autoroute a ouverte en décembre dernier jusqu à neufchâtel rn aménagée une formation de meilleure qualité adapta

In [14]:
# récupérer 30 documents aléatoires

clean_texts_aleatoires = random.sample(clean_texts, 30)

# LDA topic modeling

In [9]:
!python -m spacy download fr_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 52.0 MB/s  0:00:01m0:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')


In [10]:
nlp = spacy.load("fr_core_news_md")

In [15]:
def preprocess(text):
    
    doc = nlp(text)
    
    tokens = [
        token.lemma_
        for token in doc
        if token.is_alpha
        and not token.is_stop
        and len(token) > 3
    ]
    
    return tokens

processed_texts = [preprocess(t) for t in clean_texts_aleatoires]
processed_texts[0]

['election',
 'législatif',
 'mars',
 'claude',
 'desbons',
 'gers',
 'force',
 'candidat',
 'allianc',
 'français',
 'progrès',
 'elcessoire',
 'gersois',
 'dimanche',
 'mars',
 'aller',
 'accomplir',
 'acte',
 'important',
 'citoyen',
 'citoyen',
 'aller',
 'choisir',
 'liberté',
 'représentant',
 'assemblée',
 'national',
 'marquer',
 'attachement',
 'démocratie',
 'conforterez',
 'décision',
 'docteur',
 'jean',
 'laborde',
 'solliciter',
 'mandat',
 'confiance',
 'porte',
 'choix',
 'ami',
 'socialiste',
 'amener',
 'proposer',
 'prolonger',
 'ensemble',
 'contrat',
 'confiance',
 'avoir',
 'établir',
 'député',
 'jamais',
 'souhaiter',
 'rompre',
 'humaniste',
 'discret',
 'efficace',
 'chef',
 'entreprise',
 'réussir',
 'économie',
 'sage',
 'humain',
 'image',
 'presse',
 'véhiculer',
 'permettre',
 'aujourd',
 'présenter',
 'effectivement',
 'chef',
 'entreprise',
 'homme',
 'gauche',
 'beaucoup',
 'gersois',
 'fils',
 'agriculteur',
 'attacher',
 'racine',
 'rural',
 'décider

In [16]:
# Corpus LDA

dictionary = corpora.Dictionary(processed_texts)

dictionary.filter_extremes(
    no_below=5,      # mot doit apparaître dans 5 docs
    no_above=0.4     # supprimer mots trop fréquents
)

corpus = [dictionary.doc2bow(text) for text in processed_texts]

# Modèle LDA
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=10,
    passes=15,
    random_state=42
)

In [17]:
# voir les topics
topics = lda_model.print_topics(num_words=10)

for t in topics:
    print(t)

(0, '0.037*"charge" + 0.033*"apprentissage" + 0.032*"conseiller" + 0.032*"création" + 0.026*"lutte" + 0.023*"mesure" + 0.023*"préserver" + 0.022*"marier" + 0.022*"réforme" + 0.021*"europe"')
(1, '0.041*"parti" + 0.037*"communiste" + 0.020*"écologie" + 0.020*"changement" + 0.020*"voix" + 0.018*"progrès" + 0.018*"avoir" + 0.018*"département" + 0.017*"socialiste" + 0.017*"besoin"')
(2, '0.053*"volonté" + 0.031*"socialiste" + 0.031*"million" + 0.023*"conseiller" + 0.023*"général" + 0.023*"jeune" + 0.023*"formation" + 0.023*"responsable" + 0.023*"territoire" + 0.023*"majorité"')
(3, '0.033*"travailleur" + 0.028*"payer" + 0.027*"nature" + 0.027*"maintenir" + 0.023*"animal" + 0.023*"salaire" + 0.022*"parti" + 0.020*"devoir" + 0.020*"année" + 0.017*"vendre"')
(4, '0.024*"battre" + 0.019*"marier" + 0.015*"jean" + 0.015*"maire" + 0.015*"secteur" + 0.015*"conseil" + 0.015*"temps" + 0.015*"cher" + 0.015*"problème" + 0.015*"Monsieur"')
(5, '0.034*"souhaiter" + 0.024*"économie" + 0.021*"place" + 0.0

In [ ]:
# Visualisations


vis = pyLDAvis.gensim_models.prepare(
    lda_model,
    corpus,
    dictionary
)

pyLDAvis.display(vis)

# BERTopic

In [23]:
# nettoyage léger adéquat pour BERT
def clean_for_bertopic(text: str) -> str:
    text = text.lower()

    # bruit récurrent du corpus
    patterns_to_remove = [
        r"sciences po\s*/\s*fonds cevipof",
        r"vu[, ]+le[s]? candidat[s]?",
        r"imp\.[^\n]*",
        r"offset[^\n]*",
    ]
    for p in patterns_to_remove:
        text = re.sub(p, " ", text, flags=re.IGNORECASE)

    # symboles OCR / puces
    text = re.sub(r"[☒☐•▪■◆●«»“”„]", " ", text)

    # espaces / sauts de ligne
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

docs = [clean_for_bertopic(t) for t in texts]

print(len(docs))
print(docs[0][:800])

5936
département de seine-maritime - 12ème circonscription - scrutin du 21 mars 1993 alain le vern député maire de saint-saëns - 44 ans suppléant : docteur christian plailly maire de gaillefontaine - 44 ans chère madame, chère mademoiselle, cher monsieur, voici 5 ans vous m'avez élu député. j'ai depuis consacré tout mon temps et toute mon énergie pour être digne de votre confiance. la période électorale est trop souvent celle des divisions, des belles promesses ... jugez les actes, les faits ! nous n'avons pas tout réussi, bien sûr, mais j'ai travaillé pour rassembler et unir dans l'intérêt de notre circonscription, de ses habitants, de notre pays. dans notre circonscription, j'ai agi pour: la construction de logements (gournay, la feuillie, buchy, saint-saëns, neuville ferrières, neufchâtel, l


In [24]:
!pip install -U bertopic sentence-transformers umap-learn hdbscan scikit-learn

/opt/python/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=1869) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 41.5 MB/s  0:00:00m0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 53.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 10.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 49.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 64.4 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 39.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 31.9 MB/s  0:00:17m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 26.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 40.9 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/1

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda"
)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7085.33it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
# calcul des embedings
embeddings = embedding_model.encode(
    docs,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(embeddings.shape)   # attendu: (5936, 384)

Batches: 100%|██████████| 93/93 [00:41<00:00,  2.24it/s]

(5936, 384)


In [ ]:
## Sauvegarde des embeddings
# import numpy as np
# np.save("embeddings_1993.npy", embeddings)

In [31]:
from sklearn.feature_extraction.text import CountVectorizer
import spacy

nlp = spacy.load("fr_core_news_md")

stopwords = nlp.Defaults.stop_words

In [ ]:
# pipeline BERTopic



umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=35,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=list(stopwords),
    ngram_range=(1,2),
    min_df=10,
    max_df=0.8
)

topic_model = BERTopic(
    embedding_model=None,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    min_topic_size=40,
    calculate_probabilities=True
)

topics, probs = topic_model.fit_transform(docs, embeddings)

In [33]:
topic_info = topic_model.get_topic_info()
topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,235,-1_travailleurs_animaux_nature_nature animaux,"[travailleurs, animaux, nature, nature animaux...",[république française - élections législatives...
1,0,4132,0_france_circonscription_ans_droite,"[france, circonscription, ans, droite, cest, n...",[elections legislatives des 21 et 28 mars 1993...
2,1,692,1_ecologistes_nouvelle_socit_candidats,"[ecologistes, nouvelle, socit, candidats, huma...",[entente des ecologistes elections législative...
3,2,94,2_pollution_cration_nature_scientifiques,"[pollution, cration, nature, scientifiques, pa...",[elections legislatives mars 1993 votre candid...
4,3,69,3_maintenir_quils_travailleurs_devons,"[maintenir, quils, travailleurs, devons, entre...",[elections législatives du 21 mars 1993 pourqu...
5,4,65,4_animaux_nature_nature animaux_rassemblement ...,"[animaux, nature, nature animaux, rassemblemen...",[république française - élections législatives...
6,5,62,5_maintenir_quils_travailleurs_devons,"[maintenir, quils, travailleurs, devons, entre...",[elections législatives du 21 mars 1993 pourqu...
7,6,61,6_animaux_nature_rassemblement nature_nature a...,"[animaux, nature, rassemblement nature, nature...",[république française - élections législatives...
8,7,60,7_animaux_nature_nature animaux_rassemblement ...,"[animaux, nature, nature animaux, rassemblemen...",[république française - élections législatives...
9,8,56,8_dun_dune_ans_dfense,"[dun, dune, ans, dfense, cration, letat, famil...",[alliance populaire candidat de rassemblement ...


In [29]:
len(topic_info)

19

In [30]:
# Lire les mots d'un topic
for topic_id in topic_info["Topic"].head(10):
    if topic_id != -1:
        print(f"\n=== Topic {topic_id} ===")
        print(topic_model.get_topic(topic_id)[:10])


=== Topic 0 ===
[('vous', np.float64(0.036449142483156)), ('qui', np.float64(0.032294334707653395)), ('une', np.float64(0.02879381094680501)), ('notre', np.float64(0.026397852808488418)), ('au', np.float64(0.02612664830500734)), ('je', np.float64(0.02523622008604716)), ('par', np.float64(0.025128422619647876)), ('avec', np.float64(0.02471252280748615)), ('plus', np.float64(0.02446275409736067)), ('franais', np.float64(0.02349453001464673))]

=== Topic 1 ===
[('une', np.float64(0.043387732179502474)), ('ecologistes', np.float64(0.04135240124823747)), ('vie', np.float64(0.03488351266906225)), ('cologistes', np.float64(0.03335393291176216)), ('notre', np.float64(0.02732398393417963)), ('au', np.float64(0.02687443314404975)), ('par', np.float64(0.02670746792155134)), ('peu', np.float64(0.023831630006185735)), ('politique', np.float64(0.023666754990715338)), ('nous', np.float64(0.023604660892592437))]

=== Topic 2 ===
[('la loi', np.float64(0.12604300906493263)), ('loi', np.float64(0.11574

In [ ]:
new_topics, new_probs = topic_model.reduce_topics(
    docs,
    topics=topics,
    nr_topics=15
)

topic_info_reduced = topic_model.get_topic_info()
topic_info_reduced.head(20)

In [ ]:
fig1 = topic_model.visualize_topics()
fig1.show()
fig1.write_html("fig_intertopic_map.html")

In [ ]:
fig2 = topic_model.visualize_barchart(top_n_topics=12, n_words=8)
fig2.show()
fig2.write_html("fig_topic_barchart.html")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

topic_info_plot = topic_model.get_topic_info()
topic_info_plot = topic_info_plot[topic_info_plot["Topic"] != -1].copy()
topic_info_plot = topic_info_plot.sort_values("Count", ascending=False).head(15)

plt.figure(figsize=(10, 5))
plt.bar(topic_info_plot["Name"], topic_info_plot["Count"])
plt.xticks(rotation=70, ha="right")
plt.ylabel("Nombre de documents")
plt.title("Taille des principaux topics")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

df_topics = pd.DataFrame({
    "doc_id": range(len(docs)),
    "text": docs,
    "topic": topics,
    "topic_prob": [float(np.max(p)) if p is not None else np.nan for p in probs]
})

df_topics.head()